<div align="center">

<img src="https://raw.githubusercontent.com/winstonsmith1897/DantinoX/main/docs/images/dantinox.png" width="150" alt="DantinoX"/>

</div>

# DantinoX · 01 — Quickstart

<div align="center">

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winstonsmith1897/DantinoX/blob/main/docs/notebooks/01_quickstart.ipynb) &nbsp;
[![PyPI](https://img.shields.io/pypi/v/dantinox?color=7c3aed)](https://pypi.org/project/dantinox/) &nbsp;
[![GitHub](https://img.shields.io/badge/GitHub-DantinoX-181717?logo=github)](https://github.com/winstonsmith1897/DantinoX)

</div>

*Train and generate with DantinoX in a few lines — from the one-liner API to full architectural control.*

---

**You’ll learn**
- **Level-1** one-liner API — `dx.fit`, `dx.quick_generate`
- **Level-2** explicit API — `Paradigm`, `Trainer`, `ModelConfig`
- Attention variants — MHA · GQA · MLA · Sliding-Window
- FFN variants — SwiGLU · GELU-MLP · Mixture-of-Experts
- Norm & positional encodings — RMSNorm / LayerNorm · RoPE / learned / sinusoidal / none
- Decoding with `Generator` — greedy / top-k / nucleus / streaming

**Runtime** — GPU (T4 or better) · ~10 min

---

In [1]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
!pip install -q uv
!uv pip install --system -q -U "dantinox[data,hub,elf,benchmark]" "flax>=0.12,<0.13" "jax[cuda12]"

In [2]:
import dantinox as dx

dx.doctor()   # environment health check — jax/flax/CUDA alignment, GPU visibility

  warnings.warn(



  ████                █    █                █   █
  █   █  ███  ████  █████       ████   ███   █ █ 
  █   █ █   █ █   █   █    █    █   █ █   █   █  
  █   █ █  ██ █   █   █    █    █   █ █   █  █ █ 
  ████   ████ █   █   ██   ███  █   █  ███  █   █

  JAX/Flax transformer library  v0.4.7



DantinoX doctor
  dantinox             0.4.7
  jax                  0.9.2
  jaxlib               0.9.2
  flax                 0.12.6
  optax                0.2.8
  jax-cuda12-plugin    0.10.0
  jax-cuda12-pjrt      0.10.0
  transformers         4.44.2
  datasets             4.8.5
  devices              cuda:0
  ✗ jax-cuda12-plugin 0.10.0 vs jaxlib 0.9.2 — the CUDA plugin must match jaxlib exactly (PJRT errors otherwise); fix: pip install -U "jax[cuda12]"


{'versions': {'dantinox': '0.4.7',
  'jax': '0.9.2',
  'jaxlib': '0.9.2',
  'flax': '0.12.6',
  'optax': '0.2.8',
  'jax-cuda12-plugin': '0.10.0',
  'jax-cuda12-pjrt': '0.10.0',
  'transformers': '4.44.2',
  'datasets': '4.8.5'},
 'problems': ['jax-cuda12-plugin 0.10.0 vs jaxlib 0.9.2 — the CUDA plugin must match jaxlib exactly (PJRT errors otherwise); fix: pip install -U "jax[cuda12]"'],
 'warnings': [],
 'gpu': ['cuda:0'],
 'ok': False}

In [3]:
import os
import urllib.request

if not os.path.exists('tiny_shakespeare.txt'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        'tiny_shakespeare.txt')
    print('Downloaded tiny_shakespeare.txt')
else:
    print('tiny_shakespeare.txt already present')

tiny_shakespeare.txt already present


## Level 1 — One-liner API

`dx.fit('ar', corpus, **hyperparams)` trains a character-level AR model and returns the run directory.

In [4]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [5]:
import dantinox as dx

run_dir = dx.fit(
    'ar', 'tiny_shakespeare.txt',
    dim=256, n_heads=4, head_size=64, num_blocks=4,
    lr=3e-4, epochs=2, batch_size=16, tokenizer_type='bpe'
)
print('Checkpoint saved to:', run_dir)


  ──────────────────────────────────────────────────────────────
  ar  ·  causal
  ──────────────────────────────────────────────────────────────
  run dir       runs/20260715_115101
  parameters    4.5 M  (4,462,848)

  ── model ─────────────────────────────────────────────────────
  256-dim  ·  4h×64  ·  4 blocks  ·  vocab=1,000  ·  ctx=512
  MHA  ·  RoPE  ·  RMSNorm  ·  causal  ·  MLP(×4,SwiGLU)

  ── data ──────────────────────────────────────────────────────
  source        tiny_shakespeare.txt
  tokenizer     bpe  ·  1000 vocab
  tokens        442,223  (train 398,001  ·  val 44,222)

  ── training ──────────────────────────────────────────────────
  optimizer     adamw  ·  lr=3e-04  ·  cosine  ·  warmup=400
  batch         16
  schedule      2 epochs  ·  48 steps/epoch  ·  96 updates
  precision     fp32
  devices       1× GPU

  ──────────────────────────────────────────────────────────────



  ⚠  only 96 optimizer updates — the model will likely be undertrained; lower batch_size or raise epochs


  step 1: JIT compiling (may take 1-3 min on first run)...


  from .autonotebook import tqdm as notebook_tqdm


Epoch 1/2:   0%|          | 0/48 [00:00<?, ?it/s]

...

Epoch 2/2:  65%|██████▍   | 31/48 [00:00<00:00, 75.92it/s, loss=5.2766]

Epoch 2/2:  85%|████████▌ | 41/48 [00:00<00:00, 80.15it/s, loss=5.2766]

  Epoch 2/2  train=5.4235  val=5.3252 (ppl 205.5)  ★ best  0.6s  631.8k tok/s



  ──────────────────────────────────────────────────────────────
  training complete  ·  best val loss = 5.3252
  saved → runs/20260715_115101
  ──────────────────────────────────────────────────────────────



Checkpoint saved to: runs/20260715_115101


In [6]:
output = dx.quick_generate(run_dir, 'HAMLET:\n', max_new_tokens=200)
print(output)

 HAMLET:ĊPCGR itee, H, I ining be at wonistĊg not B we of theitherlIillin- entM earind,ĊThe the noc,erc buted, both right;ed oficksulted ther cSheveĊN uponantenousidwnnder. be;x-DUKE?ĊB toathge:Ċband not sw part afty will unad thy suĊhi kn's hows?ĊS a, myare isorroworrowrer'll haveenityMit what pr sirQUEEN,ĊĊĊ would;ĊSt!estatow at it.ĊThat:ĊU unareirst abaw stople mabntford seKINGh man thatĊ would's inqu is ' it and be a is the gotomows Bosti,Ċum wid:Ċres tw am sh, that he backence;ĊThature the c


## Level 2 — Explicit Paradigm API

Separate `ModelConfig` (architecture) and `TrainingConfig` (training) for full control.

In [7]:
model_cfg = dx.ModelConfig(
    paradigm="ar",
    dim=256, n_heads=4, num_blocks=4,
)
train_cfg = dx.TrainingConfig(lr=3e-4, epochs=1, batch_size=16)

paradigm = dx.Paradigm(model_cfg)
run_dir2 = dx.Trainer(paradigm, train_cfg).fit('tiny_shakespeare.txt')
print('Run dir:', run_dir2)


  ──────────────────────────────────────────────────────────────
  ar  ·  causal
  ──────────────────────────────────────────────────────────────
  run dir       runs/20260715_115137
  parameters    4.2 M  (4,223,744)

  ── model ─────────────────────────────────────────────────────
  256-dim  ·  4h×64  ·  4 blocks  ·  vocab=66  ·  ctx=512
  MHA  ·  RoPE  ·  RMSNorm  ·  causal  ·  MLP(×4,SwiGLU)

  ── data ──────────────────────────────────────────────────────
  source        tiny_shakespeare.txt
  tokenizer     char  ·  66 vocab
  tokens        1,115,394  (train 1,003,855  ·  val 111,539)

  ── training ──────────────────────────────────────────────────
  optimizer     adamw  ·  lr=3e-04  ·  cosine  ·  warmup=400
  batch         16
  schedule      1 epoch  ·  122 steps/epoch  ·  122 updates
  precision     fp32
  devices       1× GPU

  ──────────────────────────────────────────────────────────────



  ⚠  only 122 optimizer updates — the model will likely be undertrained; lower batch_size or raise epochs


  step 1: JIT compiling (may take 1-3 min on first run)...


Epoch 1/1:   0%|          | 0/122 [00:00<?, ?it/s]

  vram 0.0 GB used  ·  peak 1.2/30 GB (4%)


...

Epoch 1/1:  93%|█████████▎| 114/122 [00:13<00:00, 59.11it/s, loss=2.1916]

Epoch 1/1:  93%|█████████▎| 114/122 [00:13<00:00, 59.11it/s, loss=2.1843]

  Epoch 1/1  train=2.6031  val=2.1934 (ppl 9.0)  ★ best  1.5s (+12.0s compile)  675.6k tok/s



  ──────────────────────────────────────────────────────────────
  training complete  ·  best val loss = 2.1934
  saved → runs/20260715_115137
  ──────────────────────────────────────────────────────────────



Run dir: runs/20260715_115137


## Attention Variants

| `attention=` | Description |
|---|---|
| `"mha"` | Multi-Head Attention (default) |
| `"gqa"` | Grouped-Query Attention — add `kv_heads < n_heads` |
| `"mla"` | Multi-Latent Attention (DeepSeek-V2 style) |
| `"mha"` + `sliding_window=True` | Local Sliding-Window Attention |

In [8]:
import jax
from flax import nnx


def param_count(cfg):
    m = dx.Paradigm(cfg).build_model(nnx.Rngs(0))
    return sum(x.size for x in jax.tree_util.tree_leaves(nnx.state(m, nnx.Param)))

attn_configs = [
    ('MHA',               dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                         vocab_size=200, attention='mha')),
    ('GQA kv_heads=2',   dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                         vocab_size=200, attention='gqa', kv_heads=2)),
    ('GQA kv_heads=1',   dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                         vocab_size=200, attention='gqa', kv_heads=1)),
    ('MLA',               dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                         vocab_size=200, attention='mla')),
    ('SWA window=64',     dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                         vocab_size=200, attention='mha',
                                         sliding_window=True, context_window=64)),
]

for name, cfg in attn_configs:
    n = param_count(cfg)
    print(f'{name:20s}  {n/1e6:.2f}M params')

MHA                   4.26M params


GQA kv_heads=2        4.00M params


GQA kv_heads=1        3.86M params


MLA                   4.86M params
SWA window=64         4.26M params


In [9]:
# Train with GQA — fewer KV heads means faster inference and lower KV-cache memory
gqa_cfg = dx.ModelConfig(
    paradigm="ar",
    dim=256, n_heads=4, num_blocks=4,
    attention='gqa', kv_heads=2, ffn='moe', moe_latent=True, moe_latent_dim=32
)
gqa_run = dx.Trainer(
    dx.Paradigm(gqa_cfg),
    dx.TrainingConfig(lr=3e-4, epochs=1, batch_size=16, tokenizer_type='char'),
).fit('tiny_shakespeare.txt', run_dir='/tmp/dx_gqa')
print(dx.quick_generate(gqa_run, 'HAMLET:\n', max_new_tokens=100))


  ──────────────────────────────────────────────────────────────
  ar  ·  causal
  ──────────────────────────────────────────────────────────────
  run dir       /tmp/dx_gqa
  parameters    2.5 M  (2,483,584)

  ── model ─────────────────────────────────────────────────────
  256-dim  ·  4h×64  ·  4 blocks  ·  vocab=66  ·  ctx=512
  GQA(4q/2kv)  ·  RoPE  ·  RMSNorm  ·  causal  ·  MoE(4×top2)

  ── data ──────────────────────────────────────────────────────
  source        tiny_shakespeare.txt
  tokenizer     char  ·  66 vocab
  tokens        1,115,394  (train 1,003,855  ·  val 111,539)

  ── training ──────────────────────────────────────────────────
  optimizer     adamw  ·  lr=3e-04  ·  cosine  ·  warmup=400
  batch         16
  schedule      1 epoch  ·  122 steps/epoch  ·  122 updates
  precision     fp32
  devices       1× GPU

  ──────────────────────────────────────────────────────────────



  ⚠  only 122 optimizer updates — the model will likely be undertrained; lower batch_size or raise epochs


  step 1: JIT compiling (may take 1-3 min on first run)...


Epoch 1/1:   0%|          | 0/122 [00:00<?, ?it/s]

  vram 0.0 GB used  ·  peak 2.3/30 GB (8%)


...

Epoch 1/1:  95%|█████████▌| 116/122 [00:24<00:00, 26.92it/s, loss=10.5470]

Epoch 1/1:  99%|█████████▉| 121/122 [00:24<00:00, 29.91it/s, loss=10.5470]

  Epoch 1/1  train=11.2438  val=10.5604 (ppl 38575.7)  ★ best  3.9s (+21.1s compile)  256.9k tok/s



  ──────────────────────────────────────────────────────────────
  training complete  ·  best val loss = 10.5604
  saved → /tmp/dx_gqa
  ──────────────────────────────────────────────────────────────



HAMLET:
RE
N . eps L-eruain t kLnKIO tOft al pee;, chern iuern paloP nhavicO:
zoIBnD tore
IOI
h mowass:
Ap Q


## FFN Variants

| `ffn=` | Description |
|---|---|
| `"mlp"` | SwiGLU MLP (default) |
| `"mlp"` + `use_swiglu=False` | GELU-activated MLP |
| `"moe"` | Mixture-of-Experts — add `n_experts` and `top_k` |


In [10]:
ffn_configs = [
    ('SwiGLU MLP',     dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                      vocab_size=200, ffn='mlp')),
    ('GELU MLP',       dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                      vocab_size=200, ffn='mlp', use_swiglu=False)),
    ('MoE 4 experts',  dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                      vocab_size=200, ffn='moe', n_experts=4, top_k=2)),
    ('MoE 8 experts',  dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                      vocab_size=200, ffn='moe', n_experts=8, top_k=2)),
]

for name, cfg in ffn_configs:
    n = param_count(cfg)
    print(f'{name:20s}  {n/1e6:.2f}M params')


SwiGLU MLP            4.26M params


GELU MLP              3.21M params
MoE 4 experts         13.73M params


MoE 8 experts         26.35M params


## Norm & Positional Encoding Variants

| `norm=` | `pos_encoding=` | Notes |
|---|---|---|
| `"rmsnorm"` | `"rotary"` | Default — RoPE is relative, no pos embedding matrix |
| `"layernorm"` | `"learned"` | Classic BERT-style learned absolute positions |
| `"rmsnorm"` | `"absolute"` | Fixed sinusoidal (Transformer 2017) |
| `"rmsnorm"` | `"none"` | No position info — rely on attention patterns only |

In [11]:
norm_pos_configs = [
    ('RMSNorm + RoPE',       dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                             vocab_size=200, norm='rmsnorm', pos_encoding='rotary')),
    ('LayerNorm + Learned',  dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                             vocab_size=200, norm='layernorm', pos_encoding='learned')),
    ('RMSNorm + Sinusoidal', dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                             vocab_size=200, norm='rmsnorm', pos_encoding='absolute')),
    ('RMSNorm + None',       dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                             vocab_size=200, norm='rmsnorm', pos_encoding='none')),
]

for name, cfg in norm_pos_configs:
    n = param_count(cfg)
    print(f'{name:30s}  {n/1e6:.2f}M params')

RMSNorm + RoPE                  4.26M params


LayerNorm + Learned             4.39M params


RMSNorm + Sinusoidal            4.26M params
RMSNorm + None                  4.26M params


In [12]:
# Compare RMSNorm vs LayerNorm on the same task
for norm in ('rmsnorm', 'layernorm'):
    cfg = dx.ModelConfig(paradigm="ar", dim=128, n_heads=2, num_blocks=4,
                         norm=norm)
    rd = dx.Trainer(
        dx.Paradigm(cfg),
        dx.TrainingConfig(lr=3e-4, epochs=1, batch_size=16),
    ).fit('tiny_shakespeare.txt', run_dir=f'/tmp/dx_{norm}')
    print(f'{norm}: {dx.quick_generate(rd, "HAMLET:", max_new_tokens=60)[:80]}')


  ──────────────────────────────────────────────────────────────
  ar  ·  causal
  ──────────────────────────────────────────────────────────────
  run dir       /tmp/dx_rmsnorm
  parameters    1.1 M  (1,063,296)

  ── model ─────────────────────────────────────────────────────
  128-dim  ·  2h×64  ·  4 blocks  ·  vocab=66  ·  ctx=512
  MHA  ·  RoPE  ·  RMSNorm  ·  causal  ·  MLP(×4,SwiGLU)

  ── data ──────────────────────────────────────────────────────
  source        tiny_shakespeare.txt
  tokenizer     char  ·  66 vocab
  tokens        1,115,394  (train 1,003,855  ·  val 111,539)

  ── training ──────────────────────────────────────────────────
  optimizer     adamw  ·  lr=3e-04  ·  cosine  ·  warmup=400
  batch         16
  schedule      1 epoch  ·  122 steps/epoch  ·  122 updates
  precision     fp32
  devices       1× GPU

  ──────────────────────────────────────────────────────────────



  ⚠  only 122 optimizer updates — the model will likely be undertrained; lower batch_size or raise epochs


  step 1: JIT compiling (may take 1-3 min on first run)...


Epoch 1/1:   0%|          | 0/122 [00:00<?, ?it/s]

  vram 0.0 GB used  ·  peak 2.3/30 GB (8%)


...

Epoch 1/1:  95%|█████████▌| 116/122 [00:17<00:00, 56.14it/s, loss=2.4666]

Epoch 1/1:  95%|█████████▌| 116/122 [00:17<00:00, 56.14it/s, loss=2.4720]

  Epoch 1/1  train=2.9133  val=2.4774 (ppl 11.9)  ★ best  1.3s (+16.5s compile)  744.2k tok/s



  ──────────────────────────────────────────────────────────────
  training complete  ·  best val loss = 2.4774
  saved → /tmp/dx_layernorm
  ──────────────────────────────────────────────────────────────



layernorm: HAMLET:RETe .Kepse -iguain onk nKIistOOUTar pee;
A:
LAn sueFn se
MP


## Generator — Greedy / Top-k / Nucleus / Streaming

`Generator` wraps any trained model with multiple decoding strategies.

Every call also records metrics in `gen.last_stats` (tokens, seconds, tok/s — plus `ttft_s` when streaming); `verbose=True` prints them inline.

In [13]:
from dantinox.generator import Generator

model = dx.load(run_dir)         # load from run_dir (Level-1 run above)
gen   = Generator(run_dir)       # Generator handles tokenization automatically

for label, kwargs in [
    ('Greedy',          dict(greedy=True)),
    ('Top-k (k=40)',    dict(temperature=0.8, top_k=40)),
    ('Nucleus (p=0.9)', dict(temperature=0.9, top_p=0.9)),
]:
    print(f'=== {label} ===')
    print(gen.generate('HAMLET:\n', max_new_tokens=80, verbose=True, **kwargs))
    print()


=== Greedy ===


  [generate] 80 tok in 4.04s · 20 tok/s · greedy · seed=42


 HAMLET:ĊĊĊĊĊĊI:ĊĊĊI:ĊĊĊĊĊĊI:ĊĊĊĊI:ĊĊĊĊĊĊI:ĊĊĊĊĊĊI:ĊĊĊĊĊĊĊĊĊI:ĊĊĊĊĊĊĊĊĊĊI:ĊĊĊĊĊĊĊĊĊĊI:ĊĊĊ

=== Top-k (k=40) ===


  [generate] 80 tok in 6.24s · 13 tok/s · top-k=40 · seed=42


 HAMLET:ĊIRs:ĊTo of:ĊW, le, your me and m for a myĊThat to, it, the a,ĊĊThe, I s and that I thest:ĊE of's,ĊA is to m the me:ĊIit, the my my c: but to hisit:ĊPD is my b:ĊIo

=== Nucleus (p=0.9) ===


  [generate] 80 tok in 6.14s · 13 tok/s · nucleus p=0.9 · seed=42


 HAMLET:ĊIRs'e of VINCENTIO, but;, l of lth no me dlER b,ĊAnd tes, to be for,atred allowe,ĊQUEENinpe f noble to by'd and theill head of be he to doĊde love, the sir kingurnish:ĊĊHisar re cty cur be onnessou,



In [14]:
# Streaming — yields tokens one at a time
print('=== Streaming (top-k, k=50) ===')
for chunk in gen.stream('HAMLET:\n', max_new_tokens=80, temperature=0.8, top_k=50):
    print(chunk, end='', flush=True)
print()

# Streaming records latency stats — including time-to-first-token:
print(gen.last_stats)

=== Streaming (top-k, k=50) ===


Ċ

'

.

Ċ

...

Ċ

I

o

:


{'prompt_tokens': 7, 'new_tokens': 80, 'seconds': 6.758, 'tok_per_s': 11.8, 'ttft_s': 4.365, 'mode': 'stream', 'seed': 42}


## Analytical FLOPs Profile

`dx.profile` returns FLOPs breakdown without running any training.

In [15]:
for dim, blocks in [(128, 4), (256, 8), (512, 12), (768, 24)]:
    cfg   = dx.ModelConfig(dim=dim, n_heads=max(1, dim//64), head_size=64,
                           num_blocks=blocks, vocab_size=200)
    flops = dx.count_flops(cfg, seq_len=256, batch_size=4)
    n     = param_count(cfg)
    print(f'dim={dim:4d} blocks={blocks:2d}  {n/1e6:6.1f}M params  {flops.total/1e9:.2f} GFLOPs')

dim= 128 blocks= 4     1.1M params  2.47 GFLOPs
dim= 256 blocks= 8     8.5M params  18.36 GFLOPs


dim= 512 blocks=12    50.5M params  106.51 GFLOPs


dim= 768 blocks=24   226.9M params  473.83 GFLOPs


---

**Recap** — you learned:
- Level-1 (`dx.fit`, `dx.quick_generate`) and Level-2 (`Paradigm`, `Trainer`) APIs
- attention / FFN / norm / positional variants from a single `ModelConfig`
- decoding with `Generator` — and its `last_stats` latency metrics

**Next →** [02 · Discrete Diffusion](02_discrete_diffusion.ipynb) · [Open in Colab](https://colab.research.google.com/github/winstonsmith1897/DantinoX/blob/main/docs/notebooks/02_discrete_diffusion.ipynb)